In [13]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [14]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig
from src.qa_pipeline.knowledge_retriever.MixturedTripletsRetriever import MixturedGraphSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger, ReaderMetrics
from src.utils.data_structs import TripletCreator

#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [15]:
PKL_GRAPH_PATH = '../../data/pickled_graphs/testdb.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

In [16]:
length = 10355

print(len(formated_triplets))
formated_triplets = formated_triplets[:length]
print(len(formated_triplets))

10355
10355


#### 2. Задаём конфигурацию графа знаний

In [17]:
# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

In [ ]:
#
LANGUAGE = 'auto' # 'ru' , 'en', 'auto

#
RETRIEVER_NAME = 'mixture' # 'astar', 'bfs', 'mixture'
RETRIEVER_HYPERP = MixturedGraphSearchConfig() # AStarGraphSearchConfig, BFSSearchConfig, MixturedGraphSearchConfig

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [ ]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP,
            cache_config=KV_STORAGE_CONFIG),
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [18]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [19]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 4. Добавляем в граф загруженные триплеты

In [ ]:
print("uploading data to graph-storage")
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets)
print("uploading data to vector-storage")
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets)

#### 5. Q&A

In [11]:
qa_examples = [
  ("Which device is better in battery life: iPhone11 Pro Max or Xiaomi 11?",
  "Xiaomi 11"),
  ("Kayla has positive, negative or neutral opinion about video of 10PRO on 25.11.2020?",
  "Negative"),
  ("Do Jane and Jonathan have any common devices (which Jane and Jonathan both use)? If so, list common devices. Otherwise, answer 'No'.",
  "Xiaomi"),
  ("Whose opinions from Freda and Bruce about devices are most similar to Kayla's?",
  "Freda"),
  ("Which people have negative opinion about video of 10PRO on 25.11.2020?",
  "Kayla"),
  ("Which people have positive opinion about signal of Mi 10pro on 22.12.2018?",
  "Matthew"),
  ("Jessica has positive, negative or neutral opinion about signal of Apple on 22.12.2018?",
  "Negative")
  ]

In [12]:
for question in qa_examples:
    answer, info = rkg_main.answer_question(question[0])
    print("INFO: ", info)
    print("MODEL ANSWER: ", answer)
    print("TRUE ANSWER: ", question[1])
    print("=" * 35)

MODEL ANSWER:  На основании предоставленной информации, нельзя однозначно сказать, какое устройство имеет лучшую батарею. Отзывы о времени работы батареи как у iPhone 11 Pro Max, так и у Xiaomi 11 смешанные. Некоторые пользователи довольны временем работы батареи обоих устройств, в то время как другие жалуются на быструю разрядку аккумулятора.
TRUE ANSWER:  Xiaomi 11
MODEL ANSWER:  У Кайлы нейтральное мнение о видео 10PRO от 25 ноября 2020 года.
TRUE ANSWER:  Negative
MODEL ANSWER:  Yes
Common devices: Xiaomi
TRUE ANSWER:  Xiaomi
MODEL ANSWER:  Freda
TRUE ANSWER:  Freda
MODEL ANSWER:  На основании предоставленной информации, негативное мнение о видео 10PRO от 25.11.2020 имеет человек по имени Kayla.
TRUE ANSWER:  Kayla
MODEL ANSWER:  Matthew имеет положительное мнение о сигнале Mi 10pro на 22 декабря 2018 года.
TRUE ANSWER:  Matthew
MODEL ANSWER:  Информации о мнении Jessica относительно сигнала Apple от 22 декабря 2018 года нет.
TRUE ANSWER:  Negative
